# v3 DF26: Post-estimation diagnostics (free)

Reads a saved recovery run and explains *why* the surrogate OOS R^2 and the parameter CIs are what they are.

What it does:
- Loads a finished run (arrays + the collected dataset). No solver, no simulation.
- Retrains only the tiny moment surrogate on the saved dataset (seconds).
- Stores every metric under `<run_dir>/diagnostics/` and draws conclusions.

What it never does:
- It does not re-run Block 1 or the Block-2 collection.
- It needs only two files from the run: `arrays/recovery.npz` and `arrays/dataset.npz`.

Use it before committing to a longer (FULL) run, to separate cheap causes (surrogate under-training) from expensive ones (more collection) and structural ones (weak identification).

**Setup**

Makes the repo importable and configures float64 numerics (GPU on CUDA, CPU on Apple Metal).

In [ ]:
# Bootstrap: make the repo importable and define IN_COLAB / REPO_ROOT / RESULTS_ROOT. Idempotent.
import os, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    REPO_ROOT = "/content/deep-learning-corp-finance"
    if not os.path.isdir(os.path.join(REPO_ROOT, "src")):
        get_ipython().system("git clone --branch v3 --depth 1 "
                             "https://github.com/zhaoxuanwang/deep-learning-corp-finance.git " + REPO_ROOT)
except ImportError:
    IN_COLAB = False
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, "src", "v3")):
        REPO_ROOT = os.path.dirname(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Where the saved run lives. On Colab this is the Google Drive path the recovery wrote to.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = "/content/drive/MyDrive/df26_v3_outputs"
else:
    RESULTS_ROOT = os.path.join(REPO_ROOT, "outputs", "v3")

from src.v3.common.precision import configure_devices, silence_logging
silence_logging()
print("device mode:", configure_devices("auto"), "| results root:", RESULTS_ROOT)

**Point at the saved run**

- Leave `RUN_DIR` empty to auto-pick the latest run under `RESULTS_ROOT/df26_recovery`.
- Or set it to a specific run directory (the folder that contains `arrays/`).
- Running locally off Google Drive: copy that one run folder (its `arrays/` is enough) into `outputs/v3/df26_recovery/`.

In [ ]:
import glob
from src.v3.validation import diagnostics as dg

RUN_DIR = ""   # e.g. ".../df26_recovery/LARGE_20260620_...", or leave empty to auto-pick latest
if not RUN_DIR:
    base = os.path.join(RESULTS_ROOT, "df26_recovery")
    latest = os.path.join(base, "latest")
    RUN_DIR = os.path.realpath(latest) if os.path.exists(latest) else (
        sorted(glob.glob(os.path.join(base, "*")))[-1])
assert os.path.isdir(os.path.join(RUN_DIR, "arrays")), f"no arrays/ under {RUN_DIR}"
print("diagnosing run:", RUN_DIR)

# The sweep retrains the surrogate at each (passes, hidden) on the SAVED dataset (no solver,
# no simulation). On a GPU this is ~1-2 minutes total. On a CPU (Apple Metal runs float64 on
# CPU) each config is a few minutes, so shrink the grids there, e.g. (200, 600) x (32, 64).
PASSES_GRID = (200, 400, 800)
HIDDEN_GRID = (32, 64)
result = dg.run_all(RUN_DIR, passes_grid=PASSES_GRID, hidden_grid=HIDDEN_GRID, save=True)
print("metrics written to:", os.path.join(RUN_DIR, "diagnostics"))

### Per-moment and per-parameter tables

- `surrogate_oos_r2`: how well the surrogate fits each moment out-of-sample.
- `cv` / `uninformative`: moments that barely move across the box carry little information.
- `moment_r2`: end-to-end true vs fitted (Fig V1).
- `param_r2`, `bias`, `rmse`, `ci95_halfwidth`: per-parameter accuracy and CI width (Fig V2).

In [ ]:
import pandas as pd
pd.set_option("display.width", 140)
display(pd.DataFrame(result["moment_table"]))
display(pd.DataFrame(result["param_table"]))

### Surrogate sweep: is the OOS R^2 ceiling cheap or expensive?

- Baseline `(hidden=32, passes=200)` reproduces the run's own surrogate.
- If OOS R^2 climbs with passes/width, the ceiling is surrogate capacity/training (a free fix).
- If it plateaus, the dataset is the limit and only more `collect_rows` helps (with 8D diminishing returns).

In [ ]:
import matplotlib.pyplot as plt
sweep = pd.DataFrame(result["surrogate_sweep"])
display(sweep)
print(result["sweep_verdict"]["verdict"])

fig, ax = plt.subplots(figsize=(5, 3.2))
for h, g in sweep.groupby("hidden"):
    ax.plot(g["passes"], g["mean_oos_r2"], marker="o", label=f"hidden={h}")
ax.set_xlabel("surrogate passes"); ax.set_ylabel("mean OOS R^2"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Surrogate fit vs training budget (same saved dataset)")
plt.show()

### Oracle check: surrogate at the true parameters

Evaluates the surrogate at the true draws and compares to the true moments.

- `oracle_r2` close to 1 means the surrogate reproduces the truth: the loss is downstream (LM / refine / identification).
- `oracle_r2` low means the surrogate itself is the bottleneck.

In [ ]:
display(pd.DataFrame(result["oracle"]))

### Identification: which parameters are structurally weak?

From the surrogate Jacobian dm/dbeta, standardized by each moment's variation.

- `sensitivity`: how strongly any moment responds to the parameter. Small means weakly identified.
- `condition_number`: large means a near-flat direction exists in the objective.
- `weakest_direction`: the parameter mix that is least identified (no data or training fixes this).

In [ ]:
ident = result["identification"]
display(pd.DataFrame(ident["per_param_sensitivity"]))
print("condition number:", ident["condition_number"])
print("singular values  :", ident["singular_values"])
print("weakest direction:", ident["weakest_direction"])

### Conclusions

The cell below synthesizes the verdict: the surrogate ceiling type, the weak moments, and the weak parameters. Read it together with the saved `diagnostics/diagnostics.json`.

In [ ]:
v = result["sweep_verdict"]
dead = [r["moment"] for r in result["moment_table"] if r["uninformative"]]
weak = [r["param"] for r in result["param_table"] if r["weak"]]
poor_surr = [r["moment"] for r in result["moment_table"] if r["surrogate_oos_r2"] < 0.3]

print("SURROGATE CEILING")
print(f"  baseline mean OOS R^2 = {v['baseline_mean_oos_r2']}  ->  best = {v['best_mean_oos_r2']} "
      f"at {v['best_config']} (gain {v['gain']})")
print(f"  {v['verdict']}")
print("\nUNINFORMATIVE MOMENTS (surrogate OOS R^2 < 0.1 or flat):", dead or "none")
print("POORLY SURROGATED MOMENTS (OOS R^2 < 0.3):", poor_surr or "none")
print("WEAK PARAMETERS (R^2 < 0.5):", weak or "none")
print("IDENTIFICATION condition number:", result["identification"]["condition_number"])
print("\nRECOMMENDATION")
if v["gain"] > 0.10:
    print("  Cheap win first: raise surrogate passes/hidden to", v["best_config"],
          "and re-run estimation on the SAVED dataset (no recollection).")
elif v["gain"] < 0.03:
    print("  Surrogate is data-limited: more collect_rows is the lever (diminishing in 8D).")
else:
    print("  Mixed: take the cheap passes/hidden gain, then add collection.")
print("  Structurally weak parameters above will stay wide regardless of scale; that is the moment set, not compute.")